# HarmShield — Hate and Offensive Content Detection
This notebook documents the final leakage-safe training/evaluation workflow.
HateXplain is adapted into six outputs: `abusive` plus five target-community annotations.
Only `abusive` controls the final YES/NO harmful-content verdict.


## 1. Environment
Run the notebook from the project root. Install the pinned requirements if needed.

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd()
assert (PROJECT_ROOT / 'src').is_dir(), 'Open this notebook from the project root.'
print(PROJECT_ROOT)

In [ ]:
# Uncomment once in a fresh environment.
# %pip install -r requirements.txt

## 2. Dataset checks and exploratory analysis
The final 20% test split is reserved before model selection. All feature fitting, model configuration selection and threshold selection use only the 80% development data.


In [ ]:
from src.data_loader import load_dataset
df, text_col, labels = load_dataset()
print(df.shape)
df.head()


In [ ]:
%run run_eda.py

## 3. Leakage-safe model training
Each member model performs development-only cross-validation for configuration selection. A separate five-fold OOF run on a fixed full development set of the 80% development data selects six label-specific thresholds. The `abusive` threshold uses harmful-content F1 with a minimum precision preference of 0.75; target thresholds use their own label F1. The final 20% test set is untouched.


In [ ]:
%run models/member1_logistic_regression.py
%run models/member2_svm.py
%run models/member3_random_forest.py

## 4. Final artifacts and results
The comparison separates primary harmful-content precision/recall/F1/FPR from six-label multilabel metrics. Behavioural tests and primary confusion matrices are also generated.


In [ ]:
%run compare_models.py
%run generate_final_artifacts.py

In [ ]:
import pandas as pd
pd.read_csv('results/model_scores.csv')

## 5. Single-comment smoke test
The harmful verdict is controlled only by `abusive`. Target-group outputs are contextual and cannot independently create YES.


In [ ]:
from src.predictor import available_models, load_model, predict
for model_name, model_path in available_models().items():
    bundle = load_model(model_path)
    result = predict(bundle, 'you are a stupid idiot nobody likes you')
    print(model_name, result['is_harmful'], result['harmful_probability'], result['harmful_threshold'], result['target_predictions'])
